# Query from milvus to parquet

**IMPORTS**

In [ ]:
import pandas as pd

from tqdm import tqdm
from pymilvus import MilvusClient

In [ ]:
from config import Config

CONFIG = Config(DB_NAME="ms2query")

Create connection to milvus

In [ ]:
try:
    client = MilvusClient(
        uri=CONFIG.MILVUS_URI,
        token=CONFIG.MILVUS_TOKEN,
        timeout=CONFIG.MILVUS_TIMEOUT,
    )
    version_info = client.get_server_version()
    assert version_info is not None, "Failed to retrieve server version"
    print(f"Milvus client connected. Server Version: {version_info}")
except Exception as e:
    print(f"Failed to connect to Milvus server: {str(e)}")
    raise e

Define a fetcher to pull data from milvus

In [ ]:
def query_all_data_milvus(
    limit: int | None = None,
    collection_name: str = CONFIG.COLLECTION_NAME,
    show_progress: bool = True,
):
    results = []

    client.use_database(db_name=CONFIG.DB_NAME)
    if limit is None:
        iterator = client.query_iterator(
            collection_name=collection_name,
            output_fields=["*"],
        )
    else:
        iterator = client.query_iterator(
            collection_name=collection_name,
            output_fields=["*"],
            limit=limit,
        )

    pbar = tqdm(total=limit, desc="Fetching from Milvus", unit="rows") if show_progress else None

    try:
        while True:
            result = iterator.next()
            if not result:
                break

            results.extend(result)
            if pbar is not None:
                pbar.update(len(result))

            if limit is not None and len(results) >= limit:
                results = results[:limit]
                break
    finally:
        iterator.close()
        if pbar is not None:
            pbar.close()

    return pd.DataFrame(results)

### Fetch + Match data

In [ ]:
fetched_data = query_all_data_milvus()

print(f"Total records fetched from Milvus: {len(fetched_data)}")
print(f"Fetched data: {fetched_data.columns.tolist()}")

Insert to parquet for the experiment

In [ ]:
fetched_data.to_parquet(CONFIG.DATA_DIR_PATH / CONFIG.DB_NAME / "data.parquet")